# Data Inspection

Use this notebook for exploration and visualization. Keep repeatable cleaning/imputation logic in `scripts/model_cleaning.py` and run it with `scripts/clean_model_data.py`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import seaborn as sns
except ImportError:
    sns = None

ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
elif not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

SCRIPTS_DIR = ROOT / "scripts"
DATA_PATH = ROOT / "data" / "processed" / "alonhadat_features.csv"
CLEANED_PATH = ROOT / "data" / "processed" / "real_estate_cleaned.csv"
for path in [ROOT, SCRIPTS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Project root: {ROOT}")
print(f"Scripts dir: {SCRIPTS_DIR}")
print(f"Input exists: {DATA_PATH.exists()} -> {DATA_PATH}")


In [ ]:
# Human-readable VND display helpers for inspection only.
def format_billion_vnd(value):
    if pd.isna(value):
        return pd.NA
    return f"{value / 1_000_000_000:,.2f} tỷ"


def format_million_vnd(value):
    if pd.isna(value):
        return pd.NA
    return f"{value / 1_000_000:,.1f} triệu"


def add_readable_price_columns(frame):
    frame = frame.copy()
    if "price_vnd" in frame.columns:
        frame["price_billion_vnd"] = frame["price_vnd"] / 1_000_000_000
        frame["price_display"] = frame["price_vnd"].map(format_billion_vnd)
    if "price_per_m2" in frame.columns:
        frame["price_per_m2_million_vnd"] = frame["price_per_m2"] / 1_000_000
        frame["price_per_m2_display"] = frame["price_per_m2"].map(format_million_vnd)
    return frame


In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
missing_table = pd.DataFrame({"missing_count": missing, "missing_percent": missing_percent})
missing_table[missing_table["missing_count"] > 0]


In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
summary = df[numeric_cols].describe().T
summary


In [ ]:
df_plot = df.dropna(subset=["price_vnd", "area_m2"]).copy()
df_plot = df_plot[(df_plot["price_vnd"] > 0) & (df_plot["area_m2"] > 0)]
df_plot["price_per_m2"] = df_plot["price_vnd"] / df_plot["area_m2"]
df_plot["price_billion_vnd"] = df_plot["price_vnd"] / 1_000_000_000
df_plot["price_per_m2_million_vnd"] = df_plot["price_per_m2"] / 1_000_000
df_plot[["price_vnd", "price_billion_vnd", "area_m2", "price_per_m2", "price_per_m2_million_vnd"]].describe()


In [ ]:
# Easier-to-read price preview.
df_readable = add_readable_price_columns(df_plot)
readable_cols = [
    "title",
    "locality",
    "area_m2",
    "price_display",
    "price_per_m2_display",
]

df_readable[readable_cols].head(10)


In [ ]:
# Missing value chart for presentation/explanation.
missing_nonzero = missing_table[missing_table["missing_count"] > 0].sort_values("missing_percent")

if len(missing_nonzero) > 0:
    ax = missing_nonzero["missing_percent"].plot(
        kind="barh",
        figsize=(8, 6),
        title="Missing Values (%)",
    )
    ax.set_xlabel("Missing percent")
    plt.tight_layout()
else:
    print("No missing values found.")


In [ ]:
# Target distribution before modeling. Log scale helps reveal skew.
df_plot["log_price"] = np.log1p(df_plot["price_vnd"])
df_plot["log_price_per_m2"] = np.log1p(df_plot["price_per_m2"])

df_plot[["price_billion_vnd", "price_per_m2_million_vnd", "log_price", "log_price_per_m2"]].hist(
    bins=40,
    figsize=(12, 8),
)
plt.tight_layout()


In [ ]:
# Show the exact outlier thresholds used by model_cleaning.py.
lower = df_plot["price_per_m2"].quantile(0.01)
upper = df_plot["price_per_m2"].quantile(0.99)

print(f"1st percentile price_per_m2: {lower / 1_000_000:,.1f} triệu VND/m2")
print(f"99th percentile price_per_m2: {upper / 1_000_000:,.1f} triệu VND/m2")
print(f"Rows outside threshold: {((df_plot['price_per_m2'] < lower) | (df_plot['price_per_m2'] > upper)).sum()}")


In [ ]:
# Feature-target relationship checks.
feature_cols = [
    "area_m2",
    "num_bedrooms",
    "num_floors",
    "road_width_m",
    "distance_to_center_km",
]

for col in feature_cols:
    if col not in df_plot.columns:
        continue
    ax = df_plot.plot.scatter(
        x=col,
        y="price_billion_vnd",
        alpha=0.3,
        figsize=(5, 4),
        title=f"{col} vs price",
    )
    ax.set_ylabel("price (billion VND)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Categorical value counts to catch messy or rare categories.
for col in ["region", "locality", "property_type", "legal_status"]:
    if col not in df.columns:
        continue
    print()
    print(col)
    display(df[col].value_counts(dropna=False).head(20).to_frame("count"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["price_vnd", "area_m2", "price_per_m2"]):
    ax.hist(df_plot[col].dropna(), bins=40)
    ax.set_title(col)
plt.tight_layout()


In [ ]:
top_cols = ["title", "street", "locality", "area_m2", "price_display", "price_per_m2_display"]
df_readable.sort_values("price_per_m2", ascending=False)[top_cols].head(20)


In [ ]:
df_readable.sort_values("price_per_m2", ascending=True)[top_cols].head(20)


In [ ]:
if sns is not None:
    corr_cols = [col for col in numeric_cols if df[col].nunique(dropna=True) > 1]
    corr = df[corr_cols].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Correlation Matrix")
    plt.tight_layout()
else:
    print("Install seaborn for the correlation heatmap.")


In [ ]:
from model_cleaning import clean_for_modeling

cleaned = clean_for_modeling(df)
print(f"Rows: {len(df)} -> {len(cleaned)}")
print(f"Columns: {df.shape[1]} -> {cleaned.shape[1]}")
print(f"Missing values remaining: {int(cleaned.isna().sum().sum())}")
cleaned.head()


In [ ]:
# Before vs after cleaning summary.
before_after = pd.DataFrame(
    {
        "raw": [len(df), df.shape[1], int(df.isna().sum().sum())],
        "cleaned": [len(cleaned), cleaned.shape[1], int(cleaned.isna().sum().sum())],
    },
    index=["rows", "columns", "missing_values"],
)

before_after


In [ ]:
cleaned.to_csv(CLEANED_PATH, index=False, encoding="utf-8-sig")
print(CLEANED_PATH)
